# Hypothesis Testing in Python: Parametric and Non-Parametric

This notebook gives a practical, end-to-end guide to major hypothesis tests used in data science.

## Goals
- Understand null and alternative hypotheses
- Learn when to use parametric vs non-parametric tests
- Run each test in Python with interpretation

All examples are reproducible and use synthetic sample data.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.weightstats import ztest
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.contingency_tables import mcnemar
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import AnovaRM

np.random.seed(42)

def report_test(test_name, p_value, alpha=0.05):
    decision = "Reject H0" if p_value < alpha else "Fail to Reject H0"
    print(f"{test_name}: p-value = {p_value:.6f} -> {decision} (alpha={alpha})")

## 2. Quick Theory

- **Null hypothesis (H0)**: no effect / no difference / no association
- **Alternative hypothesis (H1)**: effect exists
- **p-value**: probability of observing data at least this extreme if H0 is true
- **alpha**: significance threshold (commonly 0.05)

### Parametric tests
Used when assumptions (normality, homogeneity, interval/ratio scale) are reasonable.

### Non-parametric tests
Used when parametric assumptions fail or data are ordinal / skewed / non-normal.

## 3. Assumption Checks (Commonly Used Before Tests)

In [1]:
x = np.random.normal(50, 10, 60)
y = np.random.normal(53, 11, 60)

# Normality: Shapiro-Wilk
_, p_shapiro_x = stats.shapiro(x)
_, p_shapiro_y = stats.shapiro(y)
report_test("Shapiro-Wilk (x normality)", p_shapiro_x)
report_test("Shapiro-Wilk (y normality)", p_shapiro_y)

# Equal variance: Levene
_, p_levene = stats.levene(x, y)
report_test("Levene (equal variances)", p_levene)

NameError: name 'np' is not defined

## 4. Parametric Hypothesis Tests

### 4.1 One-Sample t-test (Mean vs known value)

In [ ]:
sample = np.random.normal(loc=102, scale=15, size=40)
mu0 = 100
t_stat, p_val = stats.ttest_1samp(sample, popmean=mu0)
print(f"t-stat={t_stat:.4f}")
report_test("One-sample t-test", p_val)

### 4.2 Two-Sample Independent t-test (Welch by default choice)

In [ ]:
group_a = np.random.normal(70, 8, 50)
group_b = np.random.normal(73, 10, 45)
t_stat, p_val = stats.ttest_ind(group_a, group_b, equal_var=False)
print(f"t-stat={t_stat:.4f}")
report_test("Independent t-test (Welch)", p_val)

### 4.3 Paired t-test (Before vs After)

In [ ]:
before = np.random.normal(120, 12, 35)
after = before - np.random.normal(4, 5, 35)
t_stat, p_val = stats.ttest_rel(before, after)
print(f"t-stat={t_stat:.4f}")
report_test("Paired t-test", p_val)

### 4.4 One-Way ANOVA (3+ independent groups)

In [ ]:
g1 = np.random.normal(55, 6, 40)
g2 = np.random.normal(59, 6, 40)
g3 = np.random.normal(62, 6, 40)
f_stat, p_val = stats.f_oneway(g1, g2, g3)
print(f"F-stat={f_stat:.4f}")
report_test("One-way ANOVA", p_val)

### 4.5 Two-Way ANOVA (Main effects and interaction)

In [ ]:
n = 20
df_tw = pd.DataFrame({
    'score': np.concatenate([
        np.random.normal(65, 5, n),
        np.random.normal(70, 5, n),
        np.random.normal(68, 5, n),
        np.random.normal(74, 5, n),
    ]),
    'method': ['A'] * n + ['A'] * n + ['B'] * n + ['B'] * n,
    'gender': ['F'] * n + ['M'] * n + ['F'] * n + ['M'] * n
})

model = ols('score ~ C(method) + C(gender) + C(method):C(gender)', data=df_tw).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

### 4.6 Repeated Measures ANOVA

In [ ]:
subjects = np.arange(1, 16)
time1 = np.random.normal(50, 4, len(subjects))
time2 = time1 + np.random.normal(3, 2, len(subjects))
time3 = time2 + np.random.normal(2, 2, len(subjects))

df_rm = pd.DataFrame({
    'subject': np.repeat(subjects, 3),
    'time': ['T1', 'T2', 'T3'] * len(subjects),
    'score': np.concatenate([time1, time2, time3])
})

rm = AnovaRM(df_rm, depvar='score', subject='subject', within=['time'])
rm_result = rm.fit()
print(rm_result)

### 4.7 Z-tests for Proportions (One and Two Sample)

In [ ]:
# One-sample proportion z-test
count = 56
nobs = 100
value = 0.50
z_stat, p_val = proportions_ztest(count=count, nobs=nobs, value=value)
print(f"z-stat={z_stat:.4f}")
report_test("One-sample proportion z-test", p_val)

# Two-sample proportion z-test
counts = np.array([45, 30])
nobs = np.array([80, 75])
z_stat, p_val = proportions_ztest(count=counts, nobs=nobs)
print(f"z-stat={z_stat:.4f}")
report_test("Two-sample proportion z-test", p_val)

### 4.8 Pearson Correlation Significance Test

In [ ]:
x = np.random.normal(0, 1, 80)
y = 0.6 * x + np.random.normal(0, 0.8, 80)
r, p_val = stats.pearsonr(x, y)
print(f"Pearson r={r:.4f}")
report_test("Pearson correlation test", p_val)

### 4.9 Variance Tests (Chi-square one variance, F-test two variances)

In [ ]:
# Chi-square test for a single variance
sample = np.random.normal(20, 3.5, 35)
sigma0_sq = 9.0  # hypothesized variance
n = len(sample)
sample_var = np.var(sample, ddof=1)
chi2_stat = (n - 1) * sample_var / sigma0_sq
p_left = stats.chi2.cdf(chi2_stat, df=n - 1)
p_right = 1 - p_left
p_two_tailed = 2 * min(p_left, p_right)
print(f"Chi2-stat={chi2_stat:.4f}")
report_test("Chi-square variance test (two-tailed)", p_two_tailed)

# F-test for equality of two variances
a = np.random.normal(0, 2.0, 30)
b = np.random.normal(0, 2.8, 28)
var_a = np.var(a, ddof=1)
var_b = np.var(b, ddof=1)
f_stat = var_a / var_b
df1, df2 = len(a) - 1, len(b) - 1
p_left = stats.f.cdf(f_stat, df1, df2)
p_right = 1 - p_left
p_two_tailed = 2 * min(p_left, p_right)
print(f"F-stat={f_stat:.4f}")
report_test("Two-sample F test for variances", p_two_tailed)

## 5. Non-Parametric Hypothesis Tests

### 5.1 Mann-Whitney U Test (Independent 2-group alternative to t-test)

In [ ]:
a = np.random.exponential(scale=1.0, size=40)
b = np.random.exponential(scale=1.3, size=38)
u_stat, p_val = stats.mannwhitneyu(a, b, alternative='two-sided')
print(f"U-stat={u_stat:.4f}")
report_test("Mann-Whitney U", p_val)

### 5.2 Wilcoxon Signed-Rank Test (Paired alternative to paired t-test)

In [ ]:
before = np.random.exponential(scale=10, size=30)
after = before - np.random.normal(1.0, 1.5, 30)
w_stat, p_val = stats.wilcoxon(before, after, alternative='two-sided')
print(f"W-stat={w_stat:.4f}")
report_test("Wilcoxon signed-rank", p_val)

### 5.3 Kruskal-Wallis H Test (3+ independent groups alternative to ANOVA)

In [ ]:
g1 = np.random.exponential(1.0, 35)
g2 = np.random.exponential(1.2, 35)
g3 = np.random.exponential(1.4, 35)
h_stat, p_val = stats.kruskal(g1, g2, g3)
print(f"H-stat={h_stat:.4f}")
report_test("Kruskal-Wallis", p_val)

### 5.4 Friedman Test (Repeated-measures alternative to RM-ANOVA)

In [ ]:
cond1 = np.random.normal(50, 4, 20)
cond2 = cond1 + np.random.normal(1.5, 2, 20)
cond3 = cond2 + np.random.normal(1.0, 2, 20)
f_stat, p_val = stats.friedmanchisquare(cond1, cond2, cond3)
print(f"Friedman-stat={f_stat:.4f}")
report_test("Friedman test", p_val)

### 5.5 Spearman Rank Correlation Test

In [ ]:
x = np.random.uniform(0, 100, 60)
y = np.log1p(x) + np.random.normal(0, 0.3, 60)
rho, p_val = stats.spearmanr(x, y)
print(f"Spearman rho={rho:.4f}")
report_test("Spearman correlation test", p_val)

### 5.6 Chi-Square Goodness-of-Fit Test

In [ ]:
observed = np.array([18, 22, 20, 25, 15])
expected = np.array([20, 20, 20, 20, 20])
chi2_stat, p_val = stats.chisquare(f_obs=observed, f_exp=expected)
print(f"Chi2-stat={chi2_stat:.4f}")
report_test("Chi-square goodness-of-fit", p_val)

### 5.7 Chi-Square Test of Independence

In [ ]:
table = np.array([[35, 25, 20],
                  [20, 30, 25]])
chi2_stat, p_val, dof, expected = stats.chi2_contingency(table)
print(f"Chi2-stat={chi2_stat:.4f}, dof={dof}")
print('Expected frequencies:\n', expected)
report_test("Chi-square independence", p_val)

### 5.8 Fisher's Exact Test (2x2 contingency, small sample)

In [ ]:
table_2x2 = np.array([[1, 9],
                      [11, 3]])
oddsratio, p_val = stats.fisher_exact(table_2x2, alternative='two-sided')
print(f"Odds ratio={oddsratio:.4f}")
report_test("Fisher exact test", p_val)

### 5.9 McNemar's Test (Paired nominal 2x2)

In [ ]:
# Format: [[both_no, before_no_after_yes], [before_yes_after_no, both_yes]]
m_table = np.array([[30, 12],
                    [5, 53]])
result = mcnemar(m_table, exact=False, correction=True)
print(f"Statistic={result.statistic:.4f}")
report_test("McNemar test", result.pvalue)

### 5.10 Kolmogorov-Smirnov (One-sample and Two-sample)

In [ ]:
# One-sample KS: compare sample with normal distribution
sample = np.random.normal(0, 1, 100)
ks_stat, p_val = stats.kstest(sample, 'norm')
print(f"KS one-sample stat={ks_stat:.4f}")
report_test("One-sample KS test", p_val)

# Two-sample KS: compare two distributions
s1 = np.random.normal(0, 1, 120)
s2 = np.random.normal(0.3, 1.1, 120)
ks_stat, p_val = stats.ks_2samp(s1, s2)
print(f"KS two-sample stat={ks_stat:.4f}")
report_test("Two-sample KS test", p_val)

## 6. Test Selection Cheat Sheet

- One mean vs known value: **One-sample t-test** (or Wilcoxon signed-rank if non-normal)
- Two independent means: **Independent t-test** (or Mann-Whitney U)
- Two paired means: **Paired t-test** (or Wilcoxon signed-rank)
- 3+ independent groups: **One-way ANOVA** (or Kruskal-Wallis)
- Repeated measures 3+: **RM-ANOVA** (or Friedman)
- Categorical association: **Chi-square independence** (or Fisher exact for small 2x2)
- Correlation (linear, normal): **Pearson**; otherwise **Spearman**

Use assumption checks and domain context before finalizing test choice.